## COVID-19 Analysis
**Module:** Python (pandas, matplotlib/seaborn, plotly)<br>
**Dataset:** Johns Hopkins CSSE COVID-19 time series (CC BY 4.0, final update 10 Mar 2023)<br>
**Column profile:** DATA_PROFILE.md

# 1. Project description
**Business context.** A public-health research team wants a retrospective of the pandemic: how it spread, when the waves peaked, which countries were hit hardest, how deadly it was in different places, and what India's experience looked like. The work must handle real-world messy time-series data.

**Objective.** Reshape wide cumulative time series into tidy data, engineer epidemiological metrics, and build a visual, statistically supported story in a Jupyter notebook.

**Stakeholders.** Public-health analysts, policy makers, journalists.

---

**Key questions**

1. How many cases and deaths were recorded globally, and when did the waves peak?
2. Which countries had the most cases, deaths and highest fatality rates?
3. How did India's waves compare (Delta vs Omicron)?
4. How long after cases do deaths follow?
5. Which US states had the highest death rates per 100k?

# 2. Dataset
| File | Rows | Id columns | Date columns |
|---|---|---|---|
| `time_series_covid19_confirmed_global.csv` | 289 | `Province/State, Country/Region, Lat, Long` | 1,143 daily (`1/22/20` … `3/9/23`) |
| `time_series_covid19_deaths_global.csv` | 289 | same | same |
| `time_series_covid19_recovered_global.csv` | 274 | same | same (**abandoned after 4 Aug 2021**) |
| `time_series_covid19_confirmed_US.csv` | 3,342 | `UID, iso2, iso3, code3, FIPS, Admin2, Province_State, Country_Region, Lat, Long_, Combined_Key` | same |
| `time_series_covid19_deaths_US.csv` | 3,342 | same + `Population` | same |


**IMPORTING LIBRARIES**


In [24]:
import os, sys, json, warnings
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve, roc_curve,
                             recall_score, precision_score, fbeta_score, brier_score_loss, confusion_matrix)
from IPython.display import display

warnings.filterwarnings('ignore', category=FutureWarning)

## 3. Data cleaning


In [25]:
from IPython.display import display
import pandas as pd


def load(kind):
    path = rf"D:\Classroom DA\Projects\Covid-19_Data_Analysis\data\time_series_covid19_{kind}_global.csv"
    df = pd.read_csv(path)
    df = (
        df.drop(columns=["Province/State", "Lat", "Long"])
          .groupby("Country/Region")
          .sum(numeric_only=True)
    )
    long = df.T
    long.index = pd.to_datetime(long.index, format="%m/%d/%y")
    return long

display(load("confirmed").head(5))


Country/Region,Afghanistan,Albania,Algeria,Andorra,Angola,Antarctica,Antigua and Barbuda,Argentina,Armenia,Australia,...,Uruguay,Uzbekistan,Vanuatu,Venezuela,Vietnam,West Bank and Gaza,Winter Olympics 2022,Yemen,Zambia,Zimbabwe
2020-01-22,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2020-01-23,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,2,0,0,0,0,0
2020-01-24,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,2,0,0,0,0,0
2020-01-25,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,2,0,0,0,0,0
2020-01-26,0,0,0,0,0,0,0,0,0,4,...,0,0,0,0,2,0,0,0,0,0


**1. ISSUES: Wide format (1 column per day)**<br>
    **TREATMENT: melt / transpose to long**

In [29]:
id_cols = ["Province/State", "Country/Region", "Lat", "Long"]

df_long = df.melt(
    id_vars=id_cols,
    var_name="Date",
    value_name="Confirmed"
)

df_long["Date"] = pd.to_datetime(df_long["Date"], format="%m/%d/%y")
df_long["Confirmed"] = pd.to_numeric(df_long["Confirmed"], errors="coerce")

# Combine provinces into one total per country and date
country_daily = (
    df_long.groupby(["Country/Region", "Date"], as_index=False)["Confirmed"]
    .sum()
)

display(country_daily.head(5))

,Country/Region,Date,Confirmed
0,Afghanistan,2020-01-22,0
1,Afghanistan,2020-01-23,0
2,Afghanistan,2020-01-24,0
3,Afghanistan,2020-01-25,0
4,Afghanistan,2020-01-26,0


**2. ISSUE: Values are cumulative**<BR>
**TREATMENT: diff() for daily new values**

In [30]:
country_daily = country_daily.sort_values(["Country/Region", "Date"]).copy()

country_daily["DailyNewConfirmed"] = (
    country_daily.groupby("Country/Region")["Confirmed"]
    .diff()
    .fillna(country_daily["Confirmed"])
)

display(country_daily.head(100))

,Country/Region,Date,Confirmed,DailyNewConfirmed
0,Afghanistan,2020-01-22,0,0.0
1,Afghanistan,2020-01-23,0,0.0
2,Afghanistan,2020-01-24,0,0.0
3,Afghanistan,2020-01-25,0,0.0
4,Afghanistan,2020-01-26,0,0.0
...,...,...,...,...
95,Afghanistan,2020-04-26,1463,133.0
96,Afghanistan,2020-04-27,1531,68.0
97,Afghanistan,2020-04-28,1703,172.0
98,Afghanistan,2020-04-29,1827,124.0


**3. ISSUE: Negative daily values (data corrections)** <BR>
**TREATMENT: clip(lower=0); count and report them**

In [31]:
negative_mask = country_daily["DailyNewConfirmed"] < 0
negative_count = int(negative_mask.sum())

print(f"Negative daily values: {negative_count}")
display(country_daily.loc[negative_mask, ["Country/Region", "Date", "DailyNewConfirmed"]])

country_daily["DailyNewConfirmed"] = country_daily["DailyNewConfirmed"].clip(lower=0)

Negative daily values: 171


,Country/Region,Date,DailyNewConfirmed
58,Afghanistan,2020-03-20,-2.0
712,Afghanistan,2022-01-03,-6.0
1109,Afghanistan,2023-02-04,-48.0
7021,Antigua and Barbuda,2020-07-03,-1.0
7331,Antigua and Barbuda,2021-05-09,-1.0
...,...,...,...
218394,Uruguay,2020-04-12,-21.0
221511,Vanuatu,2022-07-22,-7.0
221518,Vanuatu,2022-07-29,-6.0
226516,Yemen,2020-08-11,-1.0


**4. ISSUE: Countries split into provinces (China 34 rows, UK, France, Canada, Australia…)**<BR>
**TREATMENT: groupby('Country/Region').sum()**

In [32]:
country_daily = (
    df_long.groupby(["Country/Region", "Date"], as_index=False)["Confirmed"]
    .sum()
    .sort_values(["Country/Region", "Date"])
)

country_daily["DailyNewConfirmed"] = (
    country_daily.groupby("Country/Region")["Confirmed"]
    .diff()
    .fillna(country_daily["Confirmed"])
    .clip(lower=0)
)

display(country_daily.head())

,Country/Region,Date,Confirmed,DailyNewConfirmed
0,Afghanistan,2020-01-22,0,0.0
1,Afghanistan,2020-01-23,0,0.0
2,Afghanistan,2020-01-24,0,0.0
3,Afghanistan,2020-01-25,0,0.0
4,Afghanistan,2020-01-26,0,0.0


**5. ISSUE: Province/State 68% blank**<br>	
**TREATMENT: Expected: single-row countries**

In [33]:
country_structure = (
    df.groupby("Country/Region")["Province/State"]
      .agg(RowCount="size", ProvinceCount="count")
)

single_row_countries = country_structure["RowCount"].eq(1)
missing_province = df["Province/State"].isna()

print(f"Blank Province/State values: {missing_province.sum()} ({missing_province.mean():.1%})")
print(
    "Blanks in single-row countries:",
    int(df.loc[missing_province, "Country/Region"]
          .map(single_row_countries).fillna(False).sum())
)

# Label national-level rows for display; do not overwrite the source column.
df["Province/State (display)"] = df["Province/State"]
single_row_mask = df["Country/Region"].map(single_row_countries).fillna(False)
df.loc[missing_province & single_row_mask, "Province/State (display)"] = "National total"
df.loc[missing_province & ~single_row_mask, "Province/State (display)"] = "Unspecified"

display(df[["Country/Region", "Province/State", "Province/State (display)"]].head())

Blank Province/State values: 198 (68.5%)
Blanks in single-row countries: 193


C:\Users\User\AppData\Local\Temp\ipykernel_22112\1043451212.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Province/State (display)"] = df["Province/State"]


,Country/Region,Province/State,Province/State (display)
0,Afghanistan,NaN,National total
1,Albania,NaN,National total
2,Algeria,NaN,National total
3,Andorra,NaN,National total
4,Angola,NaN,National total


**6. ISSUE: Recovered stops being reported in Aug 2021 (last-day total = 0)**<BR>
**TREATMENT: Don't use it after mid-2021, and don't compute "active cases"**

In [34]:
# Recovered cases become unreliable after 4 Aug 2021.
recovered_cutoff = pd.Timestamp("2021-08-04")
recovered = load("recovered").loc[:recovered_cutoff]

print(f"Recovered data used through {recovered.index.max():%Y-%m-%d}")
display(recovered.tail())

Recovered data used through 2021-08-04


Country/Region,Afghanistan,Albania,Algeria,Andorra,Angola,Antarctica,Antigua and Barbuda,Argentina,Armenia,Australia,...,Uruguay,Uzbekistan,Vanuatu,Venezuela,Vietnam,West Bank and Gaza,Winter Olympics 2022,Yemen,Zambia,Zimbabwe
2021-07-31,82586,130222,115276,14210,37255,0,1235,4569552,219864,24093,...,373481,123164,3,290181,38734,311832,0,4185,187236,75856
2021-08-01,82586,130243,116009,14210,37397,0,1235,4581132,219986,24124,...,373636,123995,3,291556,43157,311918,0,4200,188106,76665
2021-08-02,82586,130256,116770,14296,38360,0,1235,4594873,220133,24141,...,373839,124728,3,292567,43157,312149,0,4212,188950,79420
2021-08-03,82586,130291,117557,14348,39389,0,1235,4606026,220289,24172,...,374044,125572,3,293672,50831,312289,0,4232,189341,81570
2021-08-04,82586,130314,118409,14380,39582,0,1239,4615834,220438,24203,...,374203,126377,3,294607,54332,312320,0,4251,189658,82994


**7.ISSUE: Non-country rows: Diamond Princess, MS Zaandam, Summer Olympics 2020, Winter Olympics 2022, Antarctica**<BR>
**TREATMENT: Drop for country analysis**

In [35]:
non_country_rows = [
    "Diamond Princess",
    "MS Zaandam",
    "Summer Olympics 2020",
    "Winter Olympics 2022",
    "Antarctica",
]

rows_before = len(country_daily)
country_daily = country_daily.loc[
    ~country_daily["Country/Region"].isin(non_country_rows)
].copy()

print(f"Removed {rows_before - len(country_daily):,} rows.")
print("Remaining non-country records:",
      country_daily["Country/Region"].isin(non_country_rows).sum())

Removed 5,715 rows.
Remaining non-country records: 0


**8. ISSUE: No population in the global files** <BR>
**TREATMENT: Merge UID_ISO_FIPS_LookUp_Table.csv for per-million rates**

In [36]:
lookup_url = (
    "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/"
    "master/csse_covid_19_data/UID_ISO_FIPS_LookUp_Table.csv"
)

population_lookup = pd.read_csv(lookup_url, usecols=["Country_Region", "Population"])
population_lookup["Population"] = pd.to_numeric(
    population_lookup["Population"], errors="coerce"
)

# Sum province/county populations to match the country-level case data.
population_by_country = (
    population_lookup.dropna(subset=["Country_Region"])
    .groupby("Country_Region")["Population"]
    .sum(min_count=1)
    .rename("Population")
    .reset_index()
)

country_daily = country_daily.merge(
    population_by_country,
    left_on="Country/Region",
    right_on="Country_Region",
    how="left",
    validate="many_to_one"
).drop(columns="Country_Region")

unmatched = sorted(country_daily.loc[country_daily["Population"].isna(), "Country/Region"].unique())
print("Countries without matched population data:", unmatched)

country_daily["CumulativeCasesPerMillion"] = (
    country_daily["Confirmed"] / country_daily["Population"] * 1_000_000
)
country_daily["DailyCasesPerMillion"] = (
    country_daily["DailyNewConfirmed"] / country_daily["Population"] * 1_000_000
)

display(country_daily.head())

Countries without matched population data: []


,Country/Region,Date,Confirmed,DailyNewConfirmed,Population,CumulativeCasesPerMillion,DailyCasesPerMillion
0,Afghanistan,2020-01-22,0,0.0,38928341.0,0.0,0.0
1,Afghanistan,2020-01-23,0,0.0,38928341.0,0.0,0.0
2,Afghanistan,2020-01-24,0,0.0,38928341.0,0.0,0.0
3,Afghanistan,2020-01-25,0,0.0,38928341.0,0.0,0.0
4,Afghanistan,2020-01-26,0,0.0,38928341.0,0.0,0.0


**9. ISSUE: US files: 115 rows with Population = 0 ("Unassigned", "Out of …", cruise ships)** <BR>
**TREATMENT: Exclude them from per-capita rates**

In [37]:
us_deaths_path = Path(
    r"D:\Classroom DA\Projects\Covid-19_Data_Analysis\data"
) / "time_series_covid19_deaths_US.csv"

us_deaths = pd.read_csv(us_deaths_path)
us_deaths["Population"] = pd.to_numeric(us_deaths["Population"], errors="coerce")

date_cols = [
    col for col in us_deaths.columns
    if not pd.isna(pd.to_datetime(col, format="%m/%d/%y", errors="coerce"))
]

# Exclude zero- or missing-population rows from per-capita calculations.
valid_population = us_deaths["Population"].gt(0)
print(f"Excluded from per-capita rates: {(~valid_population).sum()} rows")

valid_us_deaths = us_deaths.loc[valid_population].copy()

# Aggregate county deaths by state and date; use each county population once.
state_population = (
    valid_us_deaths.groupby("Province_State")["Population"]
    .sum()
    .rename("Population")
)

state_deaths_daily = (
    valid_us_deaths.melt(
        id_vars=["Province_State"],
        value_vars=date_cols,
        var_name="Date",
        value_name="CumulativeDeaths",
    )
)
state_deaths_daily["Date"] = pd.to_datetime(
    state_deaths_daily["Date"], format="%m/%d/%y"
)
state_deaths_daily["CumulativeDeaths"] = pd.to_numeric(
    state_deaths_daily["CumulativeDeaths"], errors="coerce"
)

state_deaths_daily = (
    state_deaths_daily.groupby(["Province_State", "Date"], as_index=False)
    ["CumulativeDeaths"].sum()
)
state_deaths_daily["Population"] = state_deaths_daily["Province_State"].map(
    state_population
)
state_deaths_daily["DeathsPer100k"] = (
    state_deaths_daily["CumulativeDeaths"]
    / state_deaths_daily["Population"] * 100_000
)

display(state_deaths_daily.tail())

Excluded from per-capita rates: 115 rows


,Province_State,Date,CumulativeDeaths,Population,DeathsPer100k
64003,Wyoming,2023-03-05,2002,578759,345.912547
64004,Wyoming,2023-03-06,2002,578759,345.912547
64005,Wyoming,2023-03-07,2004,578759,346.258114
64006,Wyoming,2023-03-08,2004,578759,346.258114
64007,Wyoming,2023-03-09,2004,578759,346.258114


**10. ISSUE: Deaths > confirmed on some dates (North Korea, Sudan, UK)**<BR>
**Reporting errors; flag them and exclude those country-days from CFR**

In [38]:
deaths_long = (
    load("deaths")
    .rename_axis(index="Date", columns="Country/Region")
    .stack()
    .rename("Deaths")
    .reset_index()
)

country_daily = country_daily.merge(
    deaths_long,
    on=["Country/Region", "Date"],
    how="left",
    validate="one_to_one",
)

country_daily["CFR_ReportingError"] = (
    country_daily["Deaths"] > country_daily["Confirmed"]
)

country_daily["CFR"] = (
    country_daily["Deaths"] / country_daily["Confirmed"] * 100
).where(
    ~country_daily["CFR_ReportingError"] & country_daily["Confirmed"].gt(0)
)

error_rows = country_daily.loc[
    country_daily["CFR_ReportingError"],
    ["Country/Region", "Date", "Confirmed", "Deaths"],
]

print(f"Country-days excluded from CFR: {len(error_rows):,}")
display(error_rows)

Country-days excluded from CFR: 302


,Country/Region,Date,Confirmed,Deaths
104856,"Korea, North",2022-05-14,1,6
104857,"Korea, North",2022-05-15,1,6
104858,"Korea, North",2022-05-16,1,6
104859,"Korea, North",2022-05-17,1,6
104860,"Korea, North",2022-05-18,1,6
...,...,...,...,...
105153,"Korea, North",2023-03-07,1,6
105154,"Korea, North",2023-03-08,1,6
105155,"Korea, North",2023-03-09,1,6
189789,Sudan,2020-03-13,0,1


**11. ISSUE: Day-of-week reporting artefacts (weekend dips)**<BR>
**Use 7-day rolling averages**

In [39]:
country_daily = country_daily.sort_values(["Country/Region", "Date"]).copy()

country_daily["DailyNewDeaths"] = (
    country_daily.groupby("Country/Region")["Deaths"]
    .diff()
    .fillna(country_daily["Deaths"])
    .clip(lower=0)
)

for column in ["DailyNewConfirmed", "DailyNewDeaths"]:
    country_daily[f"{column}_7d_avg"] = (
        country_daily.groupby("Country/Region")[column]
        .transform(lambda values: values.rolling(window=7, min_periods=1).mean())
    )

display(country_daily.head())

,Country/Region,Date,Confirmed,DailyNewConfirmed,Population,CumulativeCasesPerMillion,DailyCasesPerMillion,Deaths,CFR_ReportingError,CFR,DailyNewDeaths,DailyNewConfirmed_7d_avg,DailyNewDeaths_7d_avg
0,Afghanistan,2020-01-22,0,0.0,38928341.0,0.0,0.0,0,False,NaN,0.0,0.0,0.0
1,Afghanistan,2020-01-23,0,0.0,38928341.0,0.0,0.0,0,False,NaN,0.0,0.0,0.0
2,Afghanistan,2020-01-24,0,0.0,38928341.0,0.0,0.0,0,False,NaN,0.0,0.0,0.0
3,Afghanistan,2020-01-25,0,0.0,38928341.0,0.0,0.0,0,False,NaN,0.0,0.0,0.0
4,Afghanistan,2020-01-26,0,0.0,38928341.0,0.0,0.0,0,False,NaN,0.0,0.0,0.0
